# **Gemma 4 : Byte for byte, the most capable open models**
- Launched on 02 Apr 2026

**Industry-leading capabilities and mobile-first AI**

> - [Gemma 4](https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/) in four versatile sizes: Effective 2B (E2B), Effective 4B (E4B), 26B Mixture of Experts (MoE) and 31B Dense.

> - The entire family moves beyond simple chat to handle complex logic and agentic workflows.

> - Well-suited for reasoning, agentic workflows, coding, and multimodal understanding

### Setup

In [ ]:
# Change runtime type to T4 GPU

# Install the necessary dependencies
!pip install -U transformers torch accelerate

# 1. Delete preinstalled version
!pip uninstall -y torch torchvision torchaudio

# 2. Install CUDA 12.8 (that works well with T4 GPU)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Alternative Mount from Colab UI -> File

### Kaggle API Credentials

In [ ]:
from google.colab import userdata
import os

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

In [ ]:
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

# Getting model from Kaggle hub
# MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-26b-a4b")
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

# Load model
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto"
)

# model

In [ ]:
import torch

print(f"1. GPU model : {torch.cuda.get_device_name(0)}")
print(f"2. GPU architecture (Capability): {torch.cuda.get_device_capability(0)}")
print(f"3. PyTorch supported architecture: {torch.cuda.get_arch_list()}")

## **1. Gemma - Chat**

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

question_msg = "What is Google Universal Commerce Protocol"

# Prompt
messages = [
    {"role": "system", "content": "You are an AI expert in Agent Interoperability and Standardisation.  If you do not know say you do not know.  Answer in concise bullet points."},
    {"role": "user", "content": question_msg},
]

# Process input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True  # False
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse thinking
processor.parse_response(response)
print(180*"*")

# Console
console = Console()
parsed = processor.parse_response(response)

console.print(Panel(question_msg, title="[bold cyan]🧠 question", border_style="cyan"))

# Thinking process output (panel format)
console.print(Panel(parsed['thinking'], title="[bold cyan]🧠 Thinking Process", border_style="cyan"))

# Final answer output
console.print(Panel(parsed['content'], title="[bold green]🐘 Assistant's Response", border_style="green"))

print(parsed['content'])

## **2. Gemma - Image**

In [ ]:
# 1. Load the model and processor
processor = AutoProcessor.from_pretrained(MODEL_PATH)

### Example 1 :

In [ ]:
from PIL import Image

# 2. Load the image
# You can load from a URL or local file path
image_path = "/content/drive/MyDrive/Images/MultiAgentSystems_Parallel_pattern.png"
image_1 = Image.open(image_path)

image_1

In [ ]:
# 3. Prepare the prompt
prompt_1 = "<start_of_turn>user\n<|image|>\nYou are an AI expert in Agent Architecture and Worksflows.  If you do not know say you do not know.  Answer in concise bullet points.  Describe this image in detail.<end_of_turn>\n<start_of_turn>model\n"

# 4. Process inputs
inputs = processor(text=prompt_1, images=image_1, return_tensors="pt").to(model.device)

# 5. Generate output
# Note: Gemma 4 handles images via special tokens managed by the processor
outputs = model.generate(**inputs, max_new_tokens=500)
response = processor.decode(outputs[0], skip_special_tokens=True)

print(response)

### Example 2 :

In [ ]:
# 2. Load the image
# You can load from a URL or local file path
image_path = "/content/drive/MyDrive/Images/Google_ASP_HL_Architecture.png"
image_2 = Image.open(image_path)

image_2

In [ ]:
# 3. Prepare the prompt
prompt_2 = "<start_of_turn>user\n<|image|>\nYou are an AI expert in Agent Architecture and Worksflows.  If you do not know say you do not know.  Answer in concise bullet points.  Describe this image in detail.<end_of_turn>\n<start_of_turn>model\n"

# 4. Process inputs
inputs = processor(text=prompt_2, images=image_2, return_tensors="pt").to(model.device)

# 5. Generate output
# Note: Gemma 4 handles images via special tokens managed by the processor
outputs = model.generate(**inputs, max_new_tokens=500)
response = processor.decode(outputs[0], skip_special_tokens=True)

print(response)

### 📚 Learn More

- [Gemma 4 model card](https://ai.google.dev/gemma/docs/core/model_card_4)
- [Kaggle - Gemma 4](https://www.kaggle.com/models/google/gemma-4/)
- [Prompt with images and text using Gemma library](https://ai.google.dev/gemma/docs/core/gemma_library)

---

| Original Author |
| --- |
| [Sam Au](http://linkedin.com/in/samaujs) |